In [17]:
import pandas as pd
from pathlib import Path

base_dir = Path("/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/Clinical_data")

patient_id_file = base_dir / "MBCAPOLLO-ApolloPatientid_DATA_2026-04-17_1022.csv"

treatment_files = [
    base_dir / "MBCAPOLLO-PalliativeTreatment_Beyond2_line_DATA_LABELS_2026-04-30_1022.csv",
    base_dir / "MBCAPOLLO-PalliativeTreatment1_2_line_DATA_LABELS_2026-04-30_1021.csv"
]
# Read and clean mapping file safely
patient_map = pd.read_csv(patient_id_file, dtype=str)
patient_map.columns = patient_map.columns.str.strip()

patient_map["record_id"] = patient_map["record_id"].astype(str).str.strip()
patient_map["redcap_id"] = patient_map["redcap_id"].astype(str).str.strip()

# Remove empty/nan-like values
patient_map = patient_map[
    ~patient_map["record_id"].isin(["", "nan", "NaN", "None"]) &
    ~patient_map["redcap_id"].isin(["", "nan", "NaN", "None"])
].copy()

print("Mapping file summary")
print("--------------------")
print("Unique record_id:", patient_map["record_id"].nunique())
print("Unique redcap_id:", patient_map["redcap_id"].nunique())
print("Total mapping rows:", patient_map.shape[0])

Mapping file summary
--------------------
Unique record_id: 327
Unique redcap_id: 326
Total mapping rows: 327


In [18]:
# Check if one record_id maps to multiple redcap_id
record_to_redcap_check = (
    patient_map
    .groupby("record_id")["redcap_id"]
    .agg(lambda x: sorted(set(x)))
    .reset_index()
)

record_to_redcap_check["n_redcap_id"] = record_to_redcap_check["redcap_id"].apply(len)

record_id_multiple_redcap = record_to_redcap_check[
    record_to_redcap_check["n_redcap_id"] > 1
].copy()

print("Patient IDs / record_ids mapping to more than one redcap_id:",
      record_id_multiple_redcap.shape[0])

record_id_multiple_redcap

# Check if one redcap_id maps to multiple record_id
redcap_to_record_check = (
    patient_map
    .groupby("redcap_id")["record_id"]
    .agg(lambda x: sorted(set(x)))
    .reset_index()
)

redcap_to_record_check["n_record_id"] = redcap_to_record_check["record_id"].apply(len)

redcap_id_multiple_record = redcap_to_record_check[
    redcap_to_record_check["n_record_id"] > 1
].copy()

print("redcap_id values mapping to more than one Patient ID / record_id:",
      redcap_id_multiple_record.shape[0])

redcap_id_multiple_record

Patient IDs / record_ids mapping to more than one redcap_id: 0
redcap_id values mapping to more than one Patient ID / record_id: 1


,redcap_id,record_id,n_record_id
281,4908,"[233, 234]",2


In [19]:
# Create mapping without losing multiple redcap_ids
record_to_redcap = (
    patient_map
    .groupby("record_id")["redcap_id"]
    .agg(lambda x: ";".join(sorted(set(x))))
    .to_dict()
)

# Add redcap_id to both treatment files and print better summary
summary = []

for infile in treatment_files:
    df = pd.read_csv(infile, dtype=str)
    df.columns = df.columns.str.strip()

    if "Patient ID" not in df.columns:
        raise ValueError(f"'Patient ID' column not found in {infile.name}")

    df["Patient ID"] = df["Patient ID"].astype(str).str.strip()

    # Remove old redcap_id column if present
    if "redcap_id" in df.columns:
        df = df.drop(columns=["redcap_id"])

    # Map redcap_id using Patient ID
    redcap_series = df["Patient ID"].map(record_to_redcap)

    # Insert redcap_id immediately after Patient ID
    patient_id_position = df.columns.get_loc("Patient ID")
    df.insert(patient_id_position + 1, "redcap_id", redcap_series)

    # Save updated file
    outfile = infile.with_name(infile.stem + "_with_redcap_id.csv")
    df.to_csv(outfile, index=False)

    # Valid Patient IDs
    valid_patient_ids = df["Patient ID"].replace(
        ["", "nan", "NaN", "None"], pd.NA
    ).dropna()

    treatment_patient_ids = set(valid_patient_ids.unique())
    mapping_patient_ids = set(patient_map["record_id"].unique())

    unmatched = sorted(
        treatment_patient_ids - mapping_patient_ids,
        key=lambda x: int(x) if str(x).isdigit() else str(x)
    )

    matched = sorted(
        treatment_patient_ids & mapping_patient_ids,
        key=lambda x: int(x) if str(x).isdigit() else str(x)
    )

    # Patient IDs in this treatment file that map to multiple redcap_id values
    ambiguous_patient_ids = sorted(
        set(matched) & set(record_id_multiple_redcap["record_id"]),
        key=lambda x: int(x) if str(x).isdigit() else str(x)
    )

    # Count unique redcap IDs correctly, including semicolon-separated values
    valid_redcap_values = df["redcap_id"].replace(
        ["", "nan", "NaN", "None"], pd.NA
    ).dropna()

    exploded_redcap_ids = (
        valid_redcap_values
        .str.split(";")
        .explode()
        .dropna()
        .astype(str)
        .str.strip()
    )

    unique_redcap_count = exploded_redcap_ids.nunique()

    summary.append({
        "file": infile.name,
        "unique_patient_id": len(treatment_patient_ids),
        "matched_patient_id": len(matched),
        "unmatched_patient_id": len(unmatched),
        "ambiguous_patient_id_multiple_redcap": len(ambiguous_patient_ids),
        "unique_redcap_id_after_exploding": unique_redcap_count,
        "output_file": outfile.name
    })

    print("\nProcessed:", infile.name)
    print("Unique Patient ID:", len(treatment_patient_ids))
    print("Matched Patient ID:", len(matched))
    print("Unmatched Patient IDs:", len(unmatched))
    print("Unique redcap_id after splitting multiple IDs:", unique_redcap_count)
    print("Patient IDs with multiple redcap_id:", len(ambiguous_patient_ids))

    if unmatched:
        print("Unmatched IDs:")
        print(unmatched)

    if ambiguous_patient_ids:
        print("Patient IDs mapping to multiple redcap_id:")
        print(ambiguous_patient_ids)

    print("Saved:", outfile)

summary_df = pd.DataFrame(summary)
summary_df


Processed: MBCAPOLLO-PalliativeTreatment_Beyond2_line_DATA_LABELS_2026-04-30_1022.csv
Unique Patient ID: 265
Matched Patient ID: 262
Unmatched Patient IDs: 3
Unique redcap_id after splitting multiple IDs: 262
Patient IDs with multiple redcap_id: 0
Unmatched IDs:
['241', '261', '339']
Saved: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/Clinical_data/MBCAPOLLO-PalliativeTreatment_Beyond2_line_DATA_LABELS_2026-04-30_1022_with_redcap_id.csv

Processed: MBCAPOLLO-PalliativeTreatment1_2_line_DATA_LABELS_2026-04-30_1021.csv
Unique Patient ID: 366
Matched Patient ID: 327
Unmatched Patient IDs: 39
Unique redcap_id after splitting multiple IDs: 326
Patient IDs with multiple redcap_id: 0
Unmatched IDs:
['241', '242', '243', '261', '328', '332', '334', '335', '336', '337', '338', '339', '340', '341', '342', '343', '344', '345', '346', '347', '348', '349', '350', '351', '352', '353', '354', '355', '356', '357', '358', '359', '360', '361', '362', '363', '364', '365', '366']
Saved: /ix1/alee

,file,unique_patient_id,matched_patient_id,unmatched_patient_id,ambiguous_patient_id_multiple_redcap,unique_redcap_id_after_exploding,output_file
0,MBCAPOLLO-PalliativeTreatment_Beyond2_line_DAT...,265,262,3,0,262,MBCAPOLLO-PalliativeTreatment_Beyond2_line_DAT...
1,MBCAPOLLO-PalliativeTreatment1_2_line_DATA_LAB...,366,327,39,0,326,MBCAPOLLO-PalliativeTreatment1_2_line_DATA_LAB...


In [20]:
import pandas as pd
from pathlib import Path
import re

# ===============================
# Input files
# ===============================
base_dir = Path("/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/Clinical_data")

file_1_2 = base_dir / "MBCAPOLLO-PalliativeTreatment1_2_line_DATA_LABELS_2026-04-30_1021_with_redcap_id.csv"

file_beyond2 = base_dir / "MBCAPOLLO-PalliativeTreatment_Beyond2_line_DATA_LABELS_2026-04-30_1022_with_redcap_id.csv"

tp_excel = base_dir / "deidentified_shared_APOLLO_091225.xlsx"

# ===============================
# Output directory
# ===============================
out_final_summary = base_dir / "Summary_TreatmentLine_TPNumber_FINAL_filled.csv"
out_assigned_qc = base_dir / "Apollo_TP_assigned_to_treatment_line_QC.csv"
out_unassigned_qc = base_dir / "Apollo_TP_not_assigned_to_treatment_line_QC.csv"

In [21]:
# ===============================
# Helper functions
# ===============================

def clean_id(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip()
    if x in ["", "nan", "NaN", "None"]:
        return pd.NA
    if re.match(r"^\d+\.0$", x):
        x = x.replace(".0", "")
    return x


def clean_text(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip()
    if x in ["", "nan", "NaN", "None"]:
        return pd.NA
    x = re.sub(r"\s+", " ", x)
    return x


def ordinal(n):
    n = int(n)
    if 10 <= n % 100 <= 20:
        suffix = "th"
    else:
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix}"


def format_mdy(x):
    if pd.isna(x):
        return pd.NA
    return f"{x.month}/{x.day}/{x.year}"


def base_colname(col):
    col = str(col).strip()
    col = re.sub(r"\.\d+$", "", col)
    return col.strip()


def norm_col(col):
    col = base_colname(col)
    col = col.lower()
    col = re.sub(r"\s+", "", col)
    return col


def first_non_null_join(values):
    vals = []
    for v in values:
        if pd.notna(v):
            v = str(v).strip()
            if v and v not in ["nan", "NaN", "None"]:
                vals.append(v)
    vals = sorted(set(vals))
    if len(vals) == 0:
        return pd.NA
    return ";".join(vals)

In [22]:
# Read 1st/2nd line treatment file
df12 = pd.read_csv(file_1_2, dtype=str)
df12.columns = df12.columns.str.strip()

required_12 = [
    "Patient ID",
    "redcap_id",
    "Start date of 1st line treatment",
    "End date of first line treatment",
    "Start date 2nd line treatment",
    "End date of second line treatment"
]

missing_12 = [c for c in required_12 if c not in df12.columns]
if missing_12:
    raise ValueError(f"Missing columns in 1st/2nd line file: {missing_12}")

df12_small = df12[required_12].copy()

df12_small = df12_small.rename(columns={
    "Start date of 1st line treatment": "Start date 1st line treatment",
    "End date of first line treatment": "End date 1st line treatment",
    "Start date 2nd line treatment": "Start date 2nd line treatment",
    "End date of second line treatment": "End date 2nd line treatment"
})

df12_small["Patient ID"] = df12_small["Patient ID"].apply(clean_id)
df12_small["redcap_id"] = df12_small["redcap_id"].apply(clean_id)

df12_small = df12_small.replace(["", "nan", "NaN", "None"], pd.NA)

df12_wide = (
    df12_small
    .groupby(["Patient ID", "redcap_id"], dropna=False, as_index=False)
    .agg(first_non_null_join)
)

print("1st/2nd line file")
print("Rows:", df12_wide.shape[0])
print("Unique Patient ID:", df12_wide["Patient ID"].nunique())
print("Unique redcap_id:", df12_wide["redcap_id"].nunique())

1st/2nd line file
Rows: 366
Unique Patient ID: 366
Unique redcap_id: 326


In [23]:
# Read beyond-2nd-line treatment file and reshape wide
dfb = pd.read_csv(file_beyond2, dtype=str)
dfb.columns = dfb.columns.str.strip()

required_beyond = [
    "Patient ID",
    "redcap_id",
    "Treatment line",
    "Start date of respective treatment",
    "End date of respective treatment"
]

missing_beyond = [c for c in required_beyond if c not in dfb.columns]
if missing_beyond:
    raise ValueError(f"Missing columns in beyond-2nd-line file: {missing_beyond}")

dfb_small = dfb[required_beyond].copy()

dfb_small["Patient ID"] = dfb_small["Patient ID"].apply(clean_id)
dfb_small["redcap_id"] = dfb_small["redcap_id"].apply(clean_id)
dfb_small["Treatment line"] = pd.to_numeric(dfb_small["Treatment line"], errors="coerce")

dfb_small = dfb_small.replace(["", "nan", "NaN", "None"], pd.NA)

dfb_small = dfb_small[
    dfb_small["Patient ID"].notna() &
    dfb_small["redcap_id"].notna() &
    dfb_small["Treatment line"].notna()
].copy()

dfb_small["Treatment line"] = dfb_small["Treatment line"].astype(int)
dfb_small = dfb_small[dfb_small["Treatment line"] >= 3].copy()

dfb_collapsed = (
    dfb_small
    .groupby(["Patient ID", "redcap_id", "Treatment line"], dropna=False, as_index=False)
    .agg({
        "Start date of respective treatment": first_non_null_join,
        "End date of respective treatment": first_non_null_join
    })
)

wide_rows = []

for (patient_id, redcap_id), g in dfb_collapsed.groupby(["Patient ID", "redcap_id"], dropna=False):
    row = {
        "Patient ID": patient_id,
        "redcap_id": redcap_id
    }

    for _, r in g.sort_values("Treatment line").iterrows():
        line = int(r["Treatment line"])
        label = ordinal(line)

        row[f"Start date {label} line treatment"] = r["Start date of respective treatment"]
        row[f"End date {label} line treatment"] = r["End date of respective treatment"]

    wide_rows.append(row)

dfb_wide = pd.DataFrame(wide_rows)

print("Beyond-2nd-line file")
print("Rows:", dfb_wide.shape[0])
print("Unique Patient ID:", dfb_wide["Patient ID"].nunique())
print("Unique redcap_id:", dfb_wide["redcap_id"].nunique())


Beyond-2nd-line file
Rows: 262
Unique Patient ID: 262
Unique redcap_id: 262


In [24]:
# Merge treatment files and add blank TP columns
summary_df = pd.merge(
    df12_wide,
    dfb_wide,
    on=["Patient ID", "redcap_id"],
    how="outer"
)

line_nums = []

for col in summary_df.columns:
    m = re.search(r"Start date (\d+)(st|nd|rd|th) line treatment", col)
    if m:
        line_nums.append(int(m.group(1)))

line_nums = sorted(set(line_nums))

for line in line_nums:
    label = ordinal(line)
    tp_col = f"TP Number {label} line"
    if tp_col not in summary_df.columns:
        summary_df[tp_col] = pd.NA

print("Merged treatment summary")
print("Rows:", summary_df.shape[0])
print("Treatment lines:", line_nums)

Merged treatment summary
Rows: 366
Treatment lines: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


In [29]:
# Extract Blood obtained + TP Number from Excel
# Robust version: automatically detects header row

def norm_col2(col):
    """
    Normalize column/header names for matching.
    Handles hidden spaces, duplicate suffixes, and case.
    """
    col = str(col)
    col = col.replace("\ufeff", "")
    col = col.replace("\xa0", " ")
    col = col.strip()
    col = re.sub(r"\.\d+$", "", col)
    col = col.lower()
    col = re.sub(r"\s+", "", col)
    return col


def detect_header_row(excel_file, sheet_name, max_scan_rows=30):
    """
    Scan first rows to find the row containing REDCAP#, Blood obtained, and/or TP Number.
    """
    preview = pd.read_excel(
        excel_file,
        sheet_name=sheet_name,
        header=None,
        dtype=object,
        nrows=max_scan_rows
    )

    for i, row in preview.iterrows():
        vals = [norm_col2(v) for v in row.tolist() if pd.notna(v)]

        has_redcap = any(v in ["redcap#", "redcap", "redcapid"] for v in vals)
        has_blood = any(v == "bloodobtained" for v in vals)
        has_tp = any(v == "tpnumber" for v in vals)

        if has_redcap and (has_blood or has_tp):
            return i

    return None


sheet_names = [
    "ONGOING COLLECTIONS",
    "AUTOPSY PATIENTS",
    "SERIAL COLLECTIONS",
    "used plasma"
]

xlsx = pd.ExcelFile(tp_excel)

tp_rows = []

for sheet in sheet_names:
    if sheet not in xlsx.sheet_names:
        print(f"WARNING: Sheet not found, skipping: {sheet}")
        continue

    header_row = detect_header_row(tp_excel, sheet)

    if header_row is None:
        print(f"WARNING: Could not detect header row in sheet: {sheet}")
        continue

    print(f"\n{sheet}: detected header row = {header_row}")

    df_sheet = pd.read_excel(
        tp_excel,
        sheet_name=sheet,
        header=header_row,
        dtype=object
    )

    df_sheet.columns = [str(c).strip() for c in df_sheet.columns]

    cols = list(df_sheet.columns)
    norm_cols = [norm_col2(c) for c in cols]

    # Find REDCAP# column
    redcap_positions = [
        i for i, c in enumerate(norm_cols)
        if c in ["redcap#", "redcap", "redcapid"]
    ]

    if not redcap_positions:
        print(f"WARNING: No REDCAP# column found in sheet after header detection: {sheet}")
        print("Columns read:")
        print(cols[:20])
        continue

    redcap_col = cols[redcap_positions[0]]

    # Find all Blood obtained columns
    blood_positions = [
        i for i, c in enumerate(norm_cols)
        if c == "bloodobtained"
    ]

    if not blood_positions:
        print(f"WARNING: No Blood obtained columns found in sheet after header detection: {sheet}")
        print("Columns read:")
        print(cols[:30])
        continue

    print(f"{sheet}: REDCAP# column = {redcap_col}")
    print(f"{sheet}: Blood obtained columns found = {len(blood_positions)}")

    for blood_pos in blood_positions:
        blood_col = cols[blood_pos]

        # Find next Blood obtained column to define the block boundary
        next_blood_positions = [p for p in blood_positions if p > blood_pos]
        next_blood_pos = min(next_blood_positions) if next_blood_positions else len(cols)

        # Find TP Number to the right of this Blood obtained, before next Blood obtained
        candidate_tp_positions = [
            j for j in range(blood_pos + 1, next_blood_pos)
            if norm_cols[j] == "tpnumber"
        ]

        # Backup search within next 7 columns
        if not candidate_tp_positions:
            candidate_tp_positions = [
                j for j in range(blood_pos + 1, min(blood_pos + 8, len(cols)))
                if norm_cols[j] == "tpnumber"
            ]

        if not candidate_tp_positions:
            print(f"WARNING: No TP Number found to the right of {blood_col} in {sheet}")
            continue

        tp_col = cols[candidate_tp_positions[0]]

        temp = df_sheet[[redcap_col, blood_col, tp_col]].copy()
        temp.columns = ["redcap_id", "Blood obtained", "TP Number"]

        temp["redcap_id"] = temp["redcap_id"].apply(clean_id)
        temp["Blood obtained"] = pd.to_datetime(temp["Blood obtained"], errors="coerce")
        temp["TP Number"] = temp["TP Number"].apply(clean_text)

        temp["source_sheet"] = sheet
        temp["blood_column"] = blood_col
        temp["tp_column"] = tp_col

        temp = temp[
            temp["redcap_id"].notna() &
            temp["Blood obtained"].notna() &
            temp["TP Number"].notna()
        ].copy()

        temp = temp[temp["TP Number"].astype(str).str.startswith("TP")].copy()

        tp_rows.append(temp)

if tp_rows:
    tp_long = pd.concat(tp_rows, ignore_index=True)
else:
    tp_long = pd.DataFrame(columns=[
        "redcap_id", "Blood obtained", "TP Number",
        "source_sheet", "blood_column", "tp_column"
    ])

tp_long = tp_long.drop_duplicates(
    subset=["redcap_id", "Blood obtained", "TP Number"]
).copy()

print("\nFinal TP extraction summary")
print("---------------------------")
print("TP records extracted:", tp_long.shape[0])
print("Unique redcap_id with TP:", tp_long["redcap_id"].nunique())
print("Unique TP Number:", tp_long["TP Number"].nunique())

tp_long.head()


ONGOING COLLECTIONS: detected header row = 1
ONGOING COLLECTIONS: REDCAP# column = REDCAP#
ONGOING COLLECTIONS: Blood obtained columns found = 14

AUTOPSY PATIENTS: detected header row = 0
AUTOPSY PATIENTS: REDCAP# column = REDCAP#
AUTOPSY PATIENTS: Blood obtained columns found = 13

SERIAL COLLECTIONS: detected header row = 0
Columns read:
['IDC/ILC', 'REDCAP #', 'BIOS #', 'ER Status', 'PR Status', 'HER2 Status', 'Baseline', 'TP Number', '4-week', 'TP Number.1', '8-week', 'TP Number.2', '12-week', 'TP Number.3', '3-month', 'TP Number.4']

used plasma: detected header row = 0
used plasma: REDCAP# column = REDCAP#
used plasma: Blood obtained columns found = 6

Final TP extraction summary
---------------------------
TP records extracted: 1077
Unique redcap_id with TP: 349
Unique TP Number: 1076


,redcap_id,Blood obtained,TP Number,source_sheet,blood_column,tp_column
0,4385,2019-12-09,TP19-M937 (10),ONGOING COLLECTIONS,Blood obtained,TP Number
1,4276,2019-09-03,TP19-M696 (8),ONGOING COLLECTIONS,Blood obtained,TP Number
2,2398,2019-05-15,TP19-M434 (4),ONGOING COLLECTIONS,Blood obtained,TP Number
3,4257,2022-08-25,TP22-M462 (8),ONGOING COLLECTIONS,Blood obtained,TP Number
4,4645,2022-06-03,TP22-M279 (4),ONGOING COLLECTIONS,Blood obtained,TP Number


In [30]:
filled_df = summary_df.copy()

# Convert treatment date columns to datetime
date_cols = [
    c for c in filled_df.columns
    if c.startswith("Start date") or c.startswith("End date")
]

for col in date_cols:
    filled_df[col] = pd.to_datetime(filled_df[col], errors="coerce")

assigned_records = []
unassigned_records = []

for idx, row in filled_df.iterrows():
    patient_id = row["Patient ID"]
    redcap_id = row["redcap_id"]

    if pd.isna(redcap_id):
        continue

    patient_tp = tp_long[tp_long["redcap_id"] == redcap_id].copy()

    if patient_tp.empty:
        continue

    for _, tp_row in patient_tp.iterrows():
        blood_date = tp_row["Blood obtained"]
        tp_number = tp_row["TP Number"]

        possible_matches = []

        for line in line_nums:
            label = ordinal(line)

            start_col = f"Start date {label} line treatment"
            end_col = f"End date {label} line treatment"

            next_label = ordinal(line + 1)
            next_start_col = f"Start date {next_label} line treatment"

            if start_col not in filled_df.columns:
                continue

            start_date = row.get(start_col, pd.NaT)
            end_date = row.get(end_col, pd.NaT)
            next_start_date = row.get(next_start_col, pd.NaT) if next_start_col in filled_df.columns else pd.NaT

            if pd.isna(start_date):
                continue

            # --------------------------------------------------
            # Rule 1: inclusive treatment interval
            # Start date <= Blood obtained date <= End date
            # --------------------------------------------------
            within_line = False

            if pd.notna(end_date):
                within_line = (
                    blood_date >= start_date and
                    blood_date <= end_date
                )
            else:
                # If end date is missing:
                # assign from start date up to, but not including, next line start
                if pd.notna(next_start_date):
                    within_line = (
                        blood_date >= start_date and
                        blood_date < next_start_date
                    )
                else:
                    within_line = blood_date >= start_date

            # --------------------------------------------------
            # Rule 2: gap after current line and before next line
            # End date < Blood obtained date < Next line start date
            # --------------------------------------------------
            gap_after_line_before_next = False

            if pd.notna(end_date) and pd.notna(next_start_date):
                gap_after_line_before_next = (
                    blood_date > end_date and
                    blood_date < next_start_date
                )

            if within_line or gap_after_line_before_next:
                possible_matches.append({
                    "line": line,
                    "line_label": label,
                    "start_date": start_date,
                    "end_date": end_date,
                    "next_start_date": next_start_date,
                    "assignment_reason": (
                        "within_treatment_interval_inclusive"
                        if within_line
                        else "after_end_before_next_start"
                    )
                })

        if possible_matches:
            # If multiple matches occur, choose the match with the latest start date.
            # This handles boundary overlap when one line starts the same day another ends.
            best_match = sorted(
                possible_matches,
                key=lambda x: x["start_date"],
                reverse=True
            )[0]

            label = best_match["line_label"]
            tp_col = f"TP Number {label} line"

            existing = filled_df.at[idx, tp_col]

            existing_values = []
            if pd.notna(existing):
                existing_values = [
                    x.strip() for x in str(existing).split(";")
                    if x.strip()
                ]

            final_values = sorted(set(existing_values + [tp_number]))
            filled_df.at[idx, tp_col] = ";".join(final_values)

            assigned_records.append({
                "Patient ID": patient_id,
                "redcap_id": redcap_id,
                "Blood obtained": blood_date,
                "TP Number": tp_number,
                "assigned_line": best_match["line"],
                "assigned_line_label": label,
                "treatment_start": best_match["start_date"],
                "treatment_end": best_match["end_date"],
                "next_line_start": best_match["next_start_date"],
                "assignment_reason": best_match["assignment_reason"],
                "source_sheet": tp_row["source_sheet"]
            })

        else:
            unassigned_records.append({
                "Patient ID": patient_id,
                "redcap_id": redcap_id,
                "Blood obtained": blood_date,
                "TP Number": tp_number,
                "source_sheet": tp_row["source_sheet"],
                "reason": "not_within_inclusive_line_or_gap_before_next_line"
            })

assigned_df = pd.DataFrame(assigned_records)
unassigned_df = pd.DataFrame(unassigned_records)

print("Assigned TP records:", assigned_df.shape[0])
print("Unassigned TP records:", unassigned_df.shape[0])

if not assigned_df.empty:
    print("Unique Patient ID assigned:", assigned_df["Patient ID"].nunique())
    print("Unique redcap_id assigned:", assigned_df["redcap_id"].nunique())
    print("\nAssignment reason counts:")
    print(assigned_df["assignment_reason"].value_counts())

Assigned TP records: 985
Unassigned TP records: 15
Unique Patient ID assigned: 307
Unique redcap_id assigned: 306

Assignment reason counts:
within_treatment_interval_inclusive    938
after_end_before_next_start             47
Name: assignment_reason, dtype: int64


In [31]:
# Reorder columns
# Reorder columns
final_cols = ["Patient ID", "redcap_id"]

for line in line_nums:
    label = ordinal(line)

    for col in [
        f"Start date {label} line treatment",
        f"End date {label} line treatment",
        f"TP Number {label} line"
    ]:
        if col in filled_df.columns:
            final_cols.append(col)

extra_cols = [c for c in filled_df.columns if c not in final_cols]
filled_df = filled_df[final_cols + extra_cols]

# Convert dates back to M/D/YYYY
for col in date_cols:
    if col in filled_df.columns:
        filled_df[col] = filled_df[col].apply(format_mdy)

out_final_summary_gap = base_dir / "Summary_TreatmentLine_TPNumber_FINAL_filled_gap_assigned.csv"
out_assigned_qc_gap = base_dir / "Apollo_TP_assigned_to_treatment_line_QC_gap_assigned.csv"
out_unassigned_qc_gap = base_dir / "Apollo_TP_not_assigned_to_treatment_line_QC_gap_assigned.csv"

filled_df.to_csv(out_final_summary_gap, index=False)
assigned_df.to_csv(out_assigned_qc_gap, index=False)
unassigned_df.to_csv(out_unassigned_qc_gap, index=False)

tp_cols = [c for c in filled_df.columns if c.startswith("TP Number")]

rows_with_tp = filled_df[
    filled_df[tp_cols].notna().any(axis=1)
]

print("Saved final gap-assigned file:")
print(out_final_summary_gap)

print("\nSaved assigned QC:")
print(out_assigned_qc_gap)

print("\nSaved unassigned QC:")
print(out_unassigned_qc_gap)

print("\nFinal summary rows:", filled_df.shape[0])
print("Rows with at least one TP Number filled:", rows_with_tp.shape[0])
print("Unique Patient ID with TP filled:", rows_with_tp["Patient ID"].nunique())
print("Unique redcap_id with TP filled:", rows_with_tp["redcap_id"].nunique())

Saved final gap-assigned file:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/Clinical_data/Summary_TreatmentLine_TPNumber_FINAL_filled_gap_assigned.csv

Saved assigned QC:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/Clinical_data/Apollo_TP_assigned_to_treatment_line_QC_gap_assigned.csv

Saved unassigned QC:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/Clinical_data/Apollo_TP_not_assigned_to_treatment_line_QC_gap_assigned.csv

Final summary rows: 366
Rows with at least one TP Number filled: 307
Unique Patient ID with TP filled: 307
Unique redcap_id with TP filled: 306


In [33]:
import pandas as pd
from pathlib import Path

file = Path("/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/Clinical_data/Summary_TreatmentLine_TPNumber_FINAL_filled_gap_assigned.csv")

df = pd.read_csv(file, dtype=str)
df.columns = df.columns.str.strip()

# Identify TP Number columns
tp_cols = [c for c in df.columns if c.startswith("TP Number")]

print("Number of TP Number columns:", len(tp_cols))
print(tp_cols)

# Count filled TP Number cells
filled_cells = df[tp_cols].notna().sum().sum()

# Count rows/patients with at least one TP Number filled
rows_with_any_tp = df[df[tp_cols].notna().any(axis=1)]

print("Filled TP Number cells:", filled_cells)
print("Rows with at least one TP Number filled:", rows_with_any_tp.shape[0])
print("Unique Patient ID with at least one TP Number:", rows_with_any_tp["Patient ID"].nunique())
print("Unique redcap_id with at least one TP Number:", rows_with_any_tp["redcap_id"].nunique())

Number of TP Number columns: 17
['TP Number 1st line', 'TP Number 2nd line', 'TP Number 3rd line', 'TP Number 4th line', 'TP Number 5th line', 'TP Number 6th line', 'TP Number 7th line', 'TP Number 8th line', 'TP Number 9th line', 'TP Number 10th line', 'TP Number 11th line', 'TP Number 12th line', 'TP Number 13th line', 'TP Number 14th line', 'TP Number 15th line', 'TP Number 16th line', 'TP Number 17th line']
Filled TP Number cells: 708
Rows with at least one TP Number filled: 307
Unique Patient ID with at least one TP Number: 307
Unique redcap_id with at least one TP Number: 306
